In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, ArrayType, MapType, DoubleType
from pyspark.sql import functions as F

In [ ]:
try:
    spark.stop()
except:
    pass

In [ ]:
spark = (SparkSession.builder.appName("HealthcareDataProcessing_practitioner")
.config("spark.sql.files.ignoreCorruptFiles", "true")
.config("spark.driver.memory", "4g") 
.config("spark.executor.memory", "4g") 
.config("spark.memory.offHeap.enabled", "true") 
.config("spark.memory.offHeap.size", "2g") 
.master("local[*]")
.getOrCreate())

In [ ]:
bronze_path = "../../data_lake/bronze/practitioner_data/"
silver_base_path = "../../data_lake/silver/silver_practitioner/"

In [ ]:
general_coding_schema = StructType([
        StructField("system", StringType(), True),
        StructField("code", StringType(), True),
        StructField("display", StringType(), True)
    ])

type_schema = StructType([
        StructField("coding", ArrayType(general_coding_schema), True),
        StructField("text", StringType(), True)
    ])


telecom_schema = StructType([
        StructField("system", StringType(), True),
        StructField("value", StringType(), True),
        StructField("use", StringType(), True)
])

address_schema = StructType([
        StructField("line", ArrayType(StringType()), True),
        StructField("city", StringType(), True),
        StructField("state", StringType(), True),
        StructField("postalCode", StringType(), True),
        StructField("country", StringType(), True)
    ])

position_schema = StructType([
        StructField("longitude", DoubleType(), True),
        StructField("latitude", DoubleType(), True),
    ])

identifier_schema = StructType([
        StructField("system", StringType(), True),
        StructField("value", StringType(), True)
    ])

extension_schema = StructType([
        StructField("url", StringType(), True),
        StructField("valueInteger", StringType(), True)
])

name_schema = StructType([
        StructField("family", StringType(), True),
        StructField("given", ArrayType(StringType()), True),
        StructField("prefix", ArrayType(StringType()), True)
])

In [ ]:
def extract_telecom_value(system_value):
    matches = F.filter(
        col("telecom"),
        lambda x: x["system"] == system_value
    )
    
    first_match = F.try_element_at(matches, F.lit(1))   # null if no match
    return first_match

In [ ]:
df_practitioner = (spark.read.format("parquet").load(bronze_path))

In [ ]:
df_inter1 = (
    df_practitioner.select(
        col("resource.id").alias("practitioner_id"),
        F.conv(F.substring(col("resource.id"),1,15),16,10).cast("bigint").alias("practitioner_key"),
        from_json(col("resource.extension"), ArrayType(extension_schema)).alias("extension"),
        from_json(col("resource.identifier"), ArrayType(identifier_schema)).alias("identifier"),
        col("resource.active").alias("is_active"),
        from_json(col("resource.name"), ArrayType(name_schema)).alias("name"),
        from_json(col("resource.telecom"), ArrayType(telecom_schema)).alias("telecom"),
        from_json(col("resource.address"), ArrayType(address_schema)).alias("address"),
        col("resource.gender").alias("gender"),
        col("input_file_name")
    )
)

In [ ]:
df_inter2 = (
    df_inter1.select(
        col("practitioner_key"),
        col("practitioner_id"),
        col("identifier")[0]["value"].alias("npi"),
        col("name")[0]["prefix"][0].alias("prefix"),
        col("name")[0]["given"][0].alias("first_name"),
        col("name")[0]["family"].alias("last_name"),
        col("gender"),
        extract_telecom_value("phone")["value"].alias("phone"),
        extract_telecom_value("email")["value"].alias("email"),
        F.concat_ws(", ", col("address")[0]["line"]).alias("address_line"),
        col("address")[0]["city"].alias("city"),
        col("address")[0]["state"].alias("state"),
        col("address")[0]["postalCode"].alias("postal_code"),
        col("address")[0]["country"].alias("country"),
        col("is_active"),
        col("extension")[0]["valueInteger"].alias("utilization_encounters"),
        col("input_file_name"),
        current_timestamp().alias("ingestion_timestamp")
    )

)

In [ ]:
df_inter2.write.mode("overwrite").format("parquet").save(silver_base_path)

In [ ]:
spark.stop()